# Project 2

Welcome to the second NLP1 project. The goal of the second tutorial was to get you familiar with using pre-trained embeddings and moving towards a compositional encoder for text classification.

Given the skills you have developed in the last week, you will now be applying them within the dataset and task you have chosen during Project 1. You will be applying hands-on the experiments you have already explored in the Tutorial 2, and taking a deeper dive: extending your experiments, trying different set ups, possibly considering different evaluation techniques and errors analyses, and tackling your own research questions. 


The week after the deadline, your TA will perform a short oral assessment of the skills and knowledge you acquired (make sure to check the official schedule for this), probing you to answer questions about what you learnt, about what you did and about why you did the things you did (as opposed to some reasonable alternatives) as well as pros and cons of your method, and so on. 

# Get to work!

1. Find a word embedding model that is appropriate for your dataset. Think about the language, the kind of text you are working with. Find evidence that your choice is a good one: 
    * Can you tokenize your texts in a way coherent with the definition of a token in the embedding model?
    * Does the model's vocabulary give enough coverage to the words in your texts? etc.
2. Explore what your chosen word embedding model captures. Some things you might consider: 
    * Is it good at representing interesting similarities (re to your choice of task)? J
    * Have you found task-specific evidence that it's better than some other reasonable alternative?
    * Do you know about limitations of this model?
    * Any limitations that you find particularly troubling for your task? 
3. Experiment with the embed-and-pool approach.  
    * What design choices have meaningful impact in classification performance (eg, embedding size, architecture design, optimisation parameters), if any?
    * Is this better than what you could achieve last week? In other words, did pretrained embeddings add to your design?
4. Experiment with a recurrent encoder (GRU or LSTM). 
    * What design choices have meaningful impact in classification performance, if any?
    * Did you investigate the impact of data size, embedding parameters (trained or not, fixed or trainable, etc)?
    * Did you struggle with over-fitting?
    * How far can this approach go? Are you able to improve meaningfully on embed-and-pool?
5. We make decisions under uncertainty all the time, are you gathering enough information to substantiate your choices? Various things can lead to small differences in results, and these differences are not always consistent. Here are some things to think about:
    * How did you structure you hyperparameter search?
    * Are you keeping an eye on experimental variance?
6. Experiment with error analysis and try to discover what characteristics of input texts are predictive of performance differences across major design choices. 
    * What analysis features did you come up with?
    * Which ones did you manage to test and how did that go?
    * What would you prioritise in the future in order to improve your systems?

Be mindful that you will be using your experiments to write up your final blog, so make sure your code is clean and neat!

If you are unsure about anything, ask your TA! They are always happy to help you! Happy coding! :)

# Tips

## Pretrained embedding models

We will post a list with some options for you. 

## Features for analysis

Features that you can interpret as an aspect of _complexity_ in light of your task's goals.

- length
- presence of rare/common words, including unknown words
- syntactic features (like parts-of-speech, syntactic complexity, syntactic voice, presence of negation, etc.)
- semantic features (eg, the average similarity of the text to training documents or some curated set of training documents, presence of words in certain semantic classes, etc.)
- task-specific features (eg, 
- model-specific features (eg, predicted label, probability assigned to predicted label, number of parameters or layers or some other model descriptor)

In [3]:
%load_ext autoreload
%autoreload 2    

In [4]:
import sys
from pathlib import Path

course_root = Path.cwd().parent
# while not (course_root / "tools").exists():
#    course_root = course_root.parent

sys.path.insert(0, str(course_root) + "/part1")
print("course_root:", course_root)

import json
import random

import numpy as np
import pandas as pd
from data import GoEmotions

ds = GoEmotions(drop_multi_labels=True, merge_train_dev=True)
ds.summary

course_root: c:\Users\akcre\NLP\nlp1-student


,Total comments,Average n tokens,Median n tokens,Max tokens
train,40856.0,15.36,15.0,47.0
test,4590.0,15.26,15.0,37.0


In [7]:
sys.path.insert(0, str(course_root))
import torch
from part2.clslib import (
    PretrainedEmbeddings,
    collate_sequences,
    EmbedAndPoolEncoder,
    SoftmaxRegressor,
    Tokenizer,
)

emb = PretrainedEmbeddings(
    path="wiki-news-300d-1M.vec.zip",  # update path if needed
    num_dims=300,
    skip_first_line=True,
)

tok = Tokenizer(
    vocab=emb.vocab, tokenizer_path=str(course_root / "tools" / "tokenizer.perl")
)

print("Vocab size:", len(emb.vocab))
print("Embedding matrix shape:", emb.vectors.shape)

Vocab size: 830463
Embedding matrix shape: (830463, 300)


In [8]:
out = tok.tokens(ds.train["text"].tolist())
print("Sample tokenized sentences:", out[:5])

train_ids = [
    [emb.index(t) for t in toks] for toks in tok.tokens(ds.train["text"].tolist())
]
print("Sample IDs:", train_ids[:5])

Sample tokenized sentences: ['[', '"', 'My', 'favourite', 'food']
Sample IDs: [[521], [10], [1751, 2296], [3225, 9, 2456, 1774, 1857, 2076, 29, 536, 1988], [3225, 1774, 1774, 1605]]


In [10]:
embedding_matrix = torch.from_numpy(emb.vectors)

encoder = EmbedAndPoolEncoder(
    embedding_matrix=embedding_matrix,
    average_pool=True,
    padding_idx=emb.vocab.index(emb.pad_token),
    freeze_embeddings=True,
)

num_classes = ds.train["label"].nunique()  # adjust column name if different
model = SoftmaxRegressor(
    hidden_size1=128,
    hidden_size2=64,
    num_classes=num_classes,
    encoder=encoder,
    dropout=0.2,
)

# sanity check on a mini-batch of 4 examples
batch_x = collate_sequences(train_ids[:4], padding_value=encoder.pad_idx)
log_probs = model(batch_x)
print("Batch shape:", batch_x.shape, "-> Log-probs shape:", log_probs.shape)

Batch shape: torch.Size([4, 9]) -> Log-probs shape: torch.Size([4, 28])


In [11]:
from part2.clslib import LSTMEncoder

lstm_enc = LSTMEncoder(
    hidden_size=2,
    embedding_matrix=embedding_matrix,
    padding_idx=emb.vocab.index(emb.pad_token),
    freeze_embeddings=True,
)

In [12]:
lstm_enc(batch_x)

tensor([[ 0.0253, -0.2177],
        [-0.0631, -0.3619],
        [ 0.0338, -0.5593],
        [-0.0642, -0.6655]], grad_fn=<SelectBackward0>)

In [13]:
import part1
import part2
from collections import OrderedDict

In [14]:
my_seeds = [0, 42, 99]
num_folds = 3
my_cross_validation_sets = []

for seed in my_seeds:
    my_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            pairs=list(zip(ds.train["text"], ds.train["label"])), k=num_folds, seed=seed
        )
    )

emotions_outcome_space = part1.clslib.NamedClasses(names=ds.emotions)

In [15]:
results_folder = Path("./results")
results_folder.mkdir(parents=True, exist_ok=True)
sweep_csv = results_folder / "sweep.csv"

shared_opt = {"weight_decay": 1e-4, "batch_size": 256, "num_epochs": 300, "patience": 5}

sweeps = {
    "embed-and-pool": {
        "base": {
            "enc_type": "embed-and-pool",
            "enc_kwargs": {"freeze_embeddings": True},
            "regressor_kwargs": {},
            "opt_kwargs": dict(shared_opt),
        },
        "search_space": {
            "regressor_kwargs.hidden_size1": ("choice", [100, 300, 600]),
            "regressor_kwargs.hidden_size2": ("choice", [50, 100]),
            "regressor_kwargs.dropout": ("uniform", 0.0, 0.5),
            "opt_kwargs.learning_rate": ("loguniform", 3e-4, 3e-2),
        },
        "N": 40,
    },
    "lstm": {
        "base": {
            "enc_type": "lstm",
            "enc_kwargs": {"freeze_embeddings": True},
            "regressor_kwargs": {"hidden_size1": 50, "hidden_size2": 50},
            "opt_kwargs": dict(shared_opt),
        },
        "search_space": {
            "enc_kwargs.hidden_size": ("choice", [20, 50, 100]),
            "enc_kwargs.word_dropout": ("uniform", 0.0, 0.3),
            "regressor_kwargs.dropout": ("uniform", 0.0, 0.5),
            "opt_kwargs.learning_rate": ("loguniform", 3e-4, 3e-2),
        },
        "N": 30,
    },
}


def sample(spec, rng):
    kind = spec[0]
    if kind == "choice":
        return rng.choice(spec[1])
    if kind == "uniform":
        return round(rng.uniform(spec[1], spec[2]), 2)
    if kind == "loguniform":
        return float(f"{10 ** rng.uniform(np.log10(spec[1]), np.log10(spec[2])):.2g}")
    raise ValueError(kind)


def make_config(base, params):
    cfg = json.loads(json.dumps(base))
    for key, value in params.items():
        group, name = key.split(".")
        cfg[group][name] = value
    return cfg


def config_name(enc_type, params):
    return (
        enc_type + "_" + "_".join(f"{k.split('.')[1]}={v}" for k, v in params.items())
    )


done = set(pd.read_csv(sweep_csv)["name"]) if sweep_csv.exists() else set()
seed, run = my_seeds[0], 0

for enc_type, sweep in sweeps.items():
    rng = random.Random(42)
    for _ in range(sweep["N"]):
        params = {k: sample(spec, rng) for k, spec in sweep["search_space"].items()}
        name = config_name(enc_type, params)
        if name in done:
            continue
        cfg = make_config(sweep["base"], params)
        result = part2.experiment.cross_validation(
            data_splits=my_cross_validation_sets[run],
            outcome_space=emotions_outcome_space,
            tokenizer=tok,
            emb_model=emb,
            device="cuda",
            save_to=f"{results_folder}/sweep/{name}/seed-{seed}",
            **cfg,
        )
        accs = list(result["accuracies"].values())
        row = {
            "name": name,
            "enc_type": enc_type,
            **cfg["enc_kwargs"],
            **cfg["regressor_kwargs"],
            **cfg["opt_kwargs"],
            "mean_acc": np.mean(accs),
            "std_acc": np.std(accs),
        }
        df = pd.read_csv(sweep_csv) if sweep_csv.exists() else pd.DataFrame()
        pd.concat([df, pd.DataFrame([row])]).to_csv(sweep_csv, index=False)

res = pd.read_csv(sweep_csv)
res.sort_values("mean_acc", ascending=False).groupby("enc_type").head(5)

,name,enc_type,freeze_embeddings,padding_idx,hidden_size1,hidden_size2,dropout,num_classes,weight_decay,batch_size,num_epochs,patience,learning_rate,mean_acc,std_acc,hidden_size,word_dropout
44,lstm_hidden_size=100_word_dropout=0.21_dropout...,lstm,True,0,50,50,0.21,28,0.0001,256,300,5,0.00240,0.580918,0.008420,100.0,0.21
48,lstm_hidden_size=50_word_dropout=0.18_dropout=...,lstm,True,0,50,50,0.40,28,0.0001,256,300,5,0.00860,0.578911,0.003912,50.0,0.18
50,lstm_hidden_size=50_word_dropout=0.25_dropout=...,lstm,True,0,50,50,0.31,28,0.0001,256,300,5,0.01600,0.575558,0.004514,50.0,0.25
57,lstm_hidden_size=100_word_dropout=0.21_dropout...,lstm,True,0,50,50,0.11,28,0.0001,256,300,5,0.00130,0.574750,0.007217,100.0,0.21
54,lstm_hidden_size=50_word_dropout=0.11_dropout=...,lstm,True,0,50,50,0.34,28,0.0001,256,300,5,0.00760,0.572131,0.006720,50.0,0.11
22,embed-and-pool_hidden_size1=600_hidden_size2=1...,embed-and-pool,True,0,600,100,0.07,28,0.0001,256,300,5,0.00057,0.522494,0.009563,NaN,NaN
21,embed-and-pool_hidden_size1=600_hidden_size2=1...,embed-and-pool,True,0,600,100,0.11,28,0.0001,256,300,5,0.00300,0.520707,0.009677,NaN,NaN
14,embed-and-pool_hidden_size1=100_hidden_size2=1...,embed-and-pool,True,0,100,100,0.14,28,0.0001,256,300,5,0.00560,0.520168,0.007782,NaN,NaN
30,embed-and-pool_hidden_size1=300_hidden_size2=1...,embed-and-pool,True,0,300,100,0.06,28,0.0001,256,300,5,0.00220,0.519410,0.009271,NaN,NaN
17,embed-and-pool_hidden_size1=600_hidden_size2=5...,embed-and-pool,True,0,600,50,0.08,28,0.0001,256,300,5,0.00170,0.519385,0.011207,NaN,NaN


In [18]:
TRAIN = False

final_seeds = [0, 42, 99]
final_cross_validation_sets = []

for seed in final_seeds:
    final_cross_validation_sets.append(
        part1.experiment.cross_validation_split(
            pairs=list(zip(ds.train["text"], ds.train["label"])), k=5, seed=seed
        )
    )

configs = {
    "embed-and-pool": {
        "enc_type": "embed-and-pool",
        "enc_kwargs": {
            "freeze_embeddings": True,
        },
        "regressor_kwargs": {
            "hidden_size1": 600,
            "hidden_size2": 100,
            "dropout": 0.07,
        },
        "opt_kwargs": {
            "learning_rate": 5.7e-4,
            "weight_decay": 1e-4,
            "batch_size": 256,
            "num_epochs": 300,
            "patience": 5,
        },
    },
    "lstm": {
        "enc_type": "lstm",
        "enc_kwargs": {
            "freeze_embeddings": True,
            "hidden_size": 100,
            "word_dropout": 0.21,
        },
        "regressor_kwargs": {
            "hidden_size1": 50,
            "hidden_size2": 50,
            "dropout": 0.21,
        },
        "opt_kwargs": {
            "learning_rate": 2.4e-3,
            "weight_decay": 1e-4,
            "batch_size": 256,
            "num_epochs": 300,
            "patience": 5,
        },
    },
}

my_classifiers = OrderedDict()
for config in configs:
    my_classifiers[config] = []
    for run, seed in enumerate(final_seeds):
        run_folder = f"{results_folder}/{config}/seed-{seed}"
        if TRAIN or not Path(run_folder).exists():
            result = part2.experiment.cross_validation(
                data_splits=final_cross_validation_sets[run],
                outcome_space=emotions_outcome_space,
                tokenizer=tok,
                emb_model=emb,
                enc_type=configs[config]["enc_type"],
                enc_kwargs=configs[config]["enc_kwargs"],
                regressor_kwargs=configs[config]["regressor_kwargs"],
                opt_kwargs=configs[config]["opt_kwargs"],
                device="cpu",  #'cpu'
                save_to=run_folder,
            )
        else:
            result = part2.experiment.load_cross_validation_experiment(
                data_splits=final_cross_validation_sets[run],
                tokenizer=tok,
                emb_model=emb,
                load_from=run_folder,
                device="cpu",
            )
        my_classifiers[config].append(result)

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

  0%|          | 0/5 [00:00<?, ?it/s]

In [20]:
print(result.keys())

dict_keys(['models', 'predictions', 'accuracies'])


In [23]:
from IPython.display import display, Markdown

COPY = "md"   # "md" | "latex" | None

def show(table, title, index=False):
    """Pretty table in the notebook + copy-paste text version below it."""
    display(Markdown(f"**{title}**"))
    styled = table.style.format(precision=3)
    display(styled if index else styled.hide(axis="index"))
    if COPY == "md":
        print(table.to_markdown(index=index, floatfmt=".3f"), "\n")
    elif COPY == "latex":
        print(table.to_latex(index=index, float_format="%.3f", escape=True), "\n")

def clean(s):   # keep free text from breaking Markdown/LaTeX tables
    return str(s).replace("\n", " ").replace("|", "/")

# ---------- 1. overall ----------
overall = pd.DataFrame([{
    "model": m,
    "accuracy": accuracy_score(df["y"], df[m]),
    "macro-F1": f1_score(df["y"], df[m], average="macro"),
} for m in (P, L)])
show(overall, "Table 1: Out-of-fold performance (3 seeds x 5 folds)")

# ---------- 2. per-class F1 vs class frequency ----------
labels = sorted(df["y"].unique())
per_class = pd.DataFrame({
    "n": df["y"].value_counts(),
    "F1 pool": f1_score(df["y"], df[P], labels=labels, average=None),
    "F1 LSTM": f1_score(df["y"], df[L], labels=labels, average=None),
}, index=labels).assign(gap=lambda t: t["F1 LSTM"] - t["F1 pool"]) \
 .sort_values("n", ascending=False).rename_axis("emotion").reset_index()
show(per_class, "Table 2: Per-class F1, sorted by class frequency")

# ---------- 3. error types: within vs across polarity ----------
rows = []
for m in (P, L):
    wrong = df[df[m] != df["y"]]
    same = (wrong["y"].map(polarity) == wrong[m].map(polarity)).mean()
    rows.append({"model": m, "errors": len(wrong),
                 "same polarity": same, "different polarity": 1 - same})
show(pd.DataFrame(rows), "Table 3: Share of errors by polarity group")

# ---------- 4. text features ----------
rows = []
for col in ["len_bin", "negation", "contrast", "question", "exclaim",
            "placeholder", "elongation", "all_caps", "emoji", "profanity"]:
    for val, g in F.groupby(col, observed=True):
        mean, lo, hi = boot_ci(g["d"])
        rows.append({
            "feature": col, "value": str(val), "n": len(g),
            "acc pool": g["ok_pool"].mean(), "acc LSTM": g["ok_lstm"].mean(),
            "LSTM - pool": mean, "95% CI": f"[{lo:+.3f}, {hi:+.3f}]",
            "sig.": "yes" if (lo > 0 or hi < 0) else "",
        })
feature_table = pd.DataFrame(rows)
show(feature_table, "Table 4: LSTM minus pooling accuracy by text feature (bootstrap 95% CI)")

# ---------- 5. most frequent confusions ----------
conf = []
for m in (P, L):
    wrong = df[df[m] != df["y"]]
    top = (wrong.groupby(["y", m]).size().rename("count").reset_index()
                .rename(columns={"y": "true", m: "predicted"})
                .sort_values("count", ascending=False).head(10))
    top.insert(0, "model", m)
    conf.append(top)
show(pd.concat(conf), "Table 5: Ten most frequent confusions per model")

# ---------- 6. sampled disagreements ----------
dis = F[F["d"] != 0].sample(25, random_state=0)[["text", "y", P, L]].copy()
dis["text"] = dis["text"].map(clean)
dis = dis.rename(columns={"y": "true", P: "pool pred", L: "LSTM pred"})
show(dis, "Table 6: Examples where exactly one model is correct")

**Table 1: Out-of-fold performance (3 seeds x 5 folds)**

model,accuracy,macro-F1
embed-and-pool,0.525,0.334
lstm,0.581,0.394


| model          |   accuracy |   macro-F1 |
|:---------------|-----------:|-----------:|
| embed-and-pool |      0.525 |      0.334 |
| lstm           |      0.581 |      0.394 | 



**Table 2: Per-class F1, sorted by class frequency**

emotion,n,F1 pool,F1 LSTM,gap
27,43152,0.622,0.669,0.047
0,9081,0.598,0.674,0.076
4,6390,0.155,0.221,0.066
15,6204,0.835,0.892,0.057
1,5577,0.639,0.762,0.123
3,4842,0.139,0.119,-0.020
10,4839,0.160,0.197,0.036
18,4728,0.678,0.772,0.094
7,4650,0.383,0.446,0.063
2,3393,0.374,0.446,0.073


|   emotion |         n |   F1 pool |   F1 LSTM |    gap |
|----------:|----------:|----------:|----------:|-------:|
|    27.000 | 43152.000 |     0.622 |     0.669 |  0.047 |
|     0.000 |  9081.000 |     0.598 |     0.674 |  0.076 |
|     4.000 |  6390.000 |     0.155 |     0.221 |  0.066 |
|    15.000 |  6204.000 |     0.835 |     0.892 |  0.057 |
|     1.000 |  5577.000 |     0.639 |     0.762 |  0.123 |
|     3.000 |  4842.000 |     0.139 |     0.119 | -0.020 |
|    10.000 |  4839.000 |     0.160 |     0.197 |  0.036 |
|    18.000 |  4728.000 |     0.678 |     0.772 |  0.094 |
|     7.000 |  4650.000 |     0.383 |     0.446 |  0.063 |
|     2.000 |  3393.000 |     0.374 |     0.446 |  0.073 |
|    20.000 |  2934.000 |     0.469 |     0.577 |  0.108 |
|     6.000 |  2877.000 |     0.213 |     0.326 |  0.113 |
|    17.000 |  2868.000 |     0.429 |     0.520 |  0.091 |
|    25.000 |  2700.000 |     0.376 |     0.474 |  0.099 |
|    26.000 |  2439.000 |     0.344 |     0.523 |  0.178

... (8 outputs left out)

| model          |   true |   predicted |   count |
|:---------------|-------:|------------:|--------:|
| embed-and-pool |      4 |          27 |    4275 |
| embed-and-pool |     10 |          27 |    3437 |
| embed-and-pool |      3 |          27 |    3098 |
| embed-and-pool |      7 |          27 |    2441 |
| embed-and-pool |      0 |          27 |    2337 |
| embed-and-pool |      6 |          27 |    1696 |
| embed-and-pool |      2 |          27 |    1667 |
| embed-and-pool |      9 |          27 |    1532 |
| embed-and-pool |     22 |          27 |    1494 |
| embed-and-pool |      1 |          27 |    1332 |
| lstm           |      4 |          27 |    3970 |
| lstm           |     10 |          27 |    3147 |
| lstm           |      3 |          27 |    2729 |
| lstm           |      7 |          27 |    1838 |
| lstm           |      0 |          27 |    1503 |
| lstm           |     27 |           7 |    1492 |
| lstm           |     22 |          27 |    1485 |
| lstm      

**Table 6: Examples where exactly one model is correct**

text,true,pool pred,LSTM pred
Yeh 300+ years ago. Now they get paid pretty much to go to college. Nobody alive today even had great grandparents involved.,27,27,0
Tried it but meh. Uninstalled. Was super laggy,27,9,27
I just love the look of most benzos. Bars of course the most. But a handful of 10mg valiums or 2mg klonopin is also mouth watering :),18,27,18
"So funny, I just watched this one today and was struck by [NAME] in the height of his coked out glory. He is absolutely so gross.",1,27,1
"Yes, I’m aware. [NAME] loves LA and he isn’t worth 18 mil so he’d gladly take it.",27,27,18
[NAME] looking like his fucking controller disconnected,2,27,2
Amen as I dump mine this year!,27,18,27
> DJT to realize I’m not sure this is even a thing that happens.,6,27,6
The air can kill you at this temperature.,27,27,2
Bloody useless Pakistan,2,27,2


| text                                                                                                                                  |   true |   pool pred |   LSTM pred |
|:--------------------------------------------------------------------------------------------------------------------------------------|-------:|------------:|------------:|
| Yeh 300+ years ago. Now they get paid pretty much to go to college. Nobody alive today even had great grandparents involved.          |     27 |          27 |           0 |
| Tried it but meh. Uninstalled. Was super laggy                                                                                        |     27 |           9 |          27 |
| I just love the look of most benzos. Bars of course the most. But a handful of 10mg valiums or 2mg klonopin is also mouth watering :) |     18 |          27 |          18 |
| So funny, I just watched this one today and was struck by [NAME] in the height of his coked out glory. He is absolutely so 